[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/90-least-to-most/least_to_most_workbook.ipynb)

# 90 · Least-to-Most Prompting — Decompose then Solve Sequentially
## Zhou et al. 2022 — Break Hard Problems into Ordered Sub-Problems

⏱ ~90 min

Least-to-most prompting ([Zhou et al. 2022](https://arxiv.org/abs/2205.10625)) improves on CoT by first decomposing a hard problem into an ordered list of sub-problems, then solving them sequentially — each solution feeding forward as context to the next. This workbook builds the full decompose-then-solve LangGraph pipeline.

---

## Workshop Roadmap

| Part | Topic |
|------|-------|
| 1 | What least-to-most is and how it differs from standard CoT |
| 2 | The decompose node — how to prompt for ordered sub-problems |
| 3 | The solve loop — `current_idx`, solutions accumulation, `check_done` router |
| 4 | Context chaining — show the prompt each `solve_next` call sees |
| 5 | Full end-to-end run with step-by-step trace |
| 6 | Exercises |

---

## Prerequisites

- Python 3.10+
- An OpenAI API key (set as `OPENAI_API_KEY` in `.env` or Colab secrets)
- Packages: `langchain`, `langchain-openai`, `langgraph`, `python-dotenv`


In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid-looking: {key_ok}")
if not key_ok:
    print("  ACTION REQUIRED — add your key before running LLM cells.")


---
## Part 1 — What Least-to-Most Is and How It Differs from Standard CoT

Standard Chain-of-Thought (CoT) prompting asks the model to think step-by-step, but it still attempts the **full problem in one chain**. For multi-step problems requiring intermediate calculations or logical dependencies, CoT can skip steps or conflate sub-goals.

**Least-to-most prompting** adds a prior decomposition phase:

1. **Decompose**: Ask "What sub-problems must I solve first, in order?" — the LLM returns a numbered list.
2. **Solve sequentially**: Work through each sub-problem one at a time. Each call receives the full problem *plus all prior (sub-problem, answer) pairs* as context.

The key insight is that later sub-problems can **reference concrete numbers** from earlier answers, rather than re-deriving them from scratch.

| | Standard CoT | Least-to-Most |
|---|---|---|
| **Approach** | Single prompt, full chain | Decompose first, then sequential solves |
| **When it fails** | Multi-step problems with 4+ dependencies | Simple problems (overhead not worth it) |
| **When it excels** | Quick single-step reasoning | Complex arithmetic, multi-hop reasoning, planning |


In [ ]:
# Simulate least-to-most without LLM to show the data flow pattern
# In production, sub_problems come from the decompose node (LLM call)

sample_problem = "A store sells apples for $0.50 each and oranges for $0.75 each. Maria buys 4 apples and some oranges and spends $3.50 total. How many oranges did she buy?"

# These would come from the LLM decompose call:
sub_problems = [
    "What is the cost of 4 apples?",
    "How much money is left after buying 4 apples?",
    "How many oranges can Maria buy with the remaining money?"
]

# Simulate sequential solution accumulation
solutions = []
print(f"Problem: {sample_problem}\n")
print("Sub-problems (from decompose node):")
for i, sp in enumerate(sub_problems):
    print(f"  {i+1}. {sp}")

print("\nSequential solving (each solution becomes context for the next):")
mock_solutions = ["4 × $0.50 = $2.00", "$3.50 - $2.00 = $1.50 remaining", "$1.50 / $0.75 = 2 oranges"]
for i, (sp, sol) in enumerate(zip(sub_problems, mock_solutions)):
    solutions.append(sol)
    context_count = len(solutions) - 1
    print(f"\n  Step {i+1}: {sp}")
    print(f"    Context: {context_count} prior solution(s) available")
    print(f"    Answer: {sol}")

print(f"\nFinal answer derived from step {len(solutions)}: '{mock_solutions[-1]}'")


---
## Part 2 — The Decompose Node: How to Prompt for Ordered Sub-Problems

The decompose node sends a single prompt asking the LLM:

> "To solve X, what sub-problems must I solve first, in order?"

The LLM returns a numbered list. The key constraint is that **sub-problems must be ordered** — each one should build on the answers to previous ones.

### Decompose Prompt Template

```
To solve the following problem, what sub-problems must I solve first, in order?
List them as numbered steps. Each sub-problem should build on the answers to previous ones.
Output ONLY the numbered list — no extra text.

Problem: {problem}
```

Design choices:
- **Temperature = 0**: We want deterministic, logical decomposition, not creative variation.
- **"Output ONLY"**: Prevents the model from pre-solving — we want the list, not the answers.
- **"build on the answers to previous ones"**: Forces the model to think about ordering dependencies.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # deterministic decomposition

DECOMPOSE_PROMPT = '''To solve the following problem, what sub-problems must I solve first, in order?
List them as numbered steps. Each sub-problem should build on the answers to previous ones.
Output ONLY the numbered list — no extra text.

Problem: {problem}'''

def decompose(problem: str) -> list[str]:
    """Ask the LLM to decompose the problem into ordered sub-problems."""
    resp = llm.invoke([HumanMessage(content=DECOMPOSE_PROMPT.format(problem=problem))])
    lines = [l.strip() for l in resp.content.strip().split('\n') if l.strip()]
    # Strip numbering: "1. What is..." -> "What is..."
    sub_problems = []
    for line in lines:
        cleaned = line.lstrip('0123456789.)- ').strip()
        if cleaned:
            sub_problems.append(cleaned)
    return sub_problems

test_problem = "A train leaves city A at 8am traveling at 60 mph. Another train leaves city B (300 miles away) at 9am traveling at 80 mph toward city A. When do they meet, and how far from city A?"

print(f"Problem: {test_problem}\n")
sub_probs = decompose(test_problem)
print("Decomposed sub-problems:")
for i, sp in enumerate(sub_probs):
    print(f"  {i+1}. {sp}")


---
## Part 3 — The Solve Loop: `current_idx`, Solutions Accumulation, `check_done` Router

The LangGraph state holds:
- **`current_idx`**: which sub-problem we are solving next (starts at 0)
- **`solutions`**: list of answers accumulated so far (starts empty)
- **`sub_problems`**: the ordered list from the decompose node

After each `solve_next` call, the **`check_done` router** inspects `current_idx`:
- If `current_idx < len(sub_problems)` → route back to `solve_next` (continue loop)
- If `current_idx >= len(sub_problems)` → route to `finalize` → `END`

### Graph Topology

```
START → decompose_node → solve_next
                          ↓          ↑
                       check_done ──┘  (continue)
                          ↓
                        finalize → END  (done)
```


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

class LtMState(TypedDict):
    problem: str
    sub_problems: list[str]   # set by decompose node
    solutions: list[str]      # accumulated answers, one per sub-problem
    current_idx: int          # which sub-problem we are solving next
    final_answer: str

def check_done(state: LtMState) -> str:
    """Route: keep solving if more sub-problems remain, else finish."""
    if state["current_idx"] >= len(state["sub_problems"]):
        return "done"
    return "continue"

print("LtMState fields:", list(LtMState.__annotations__.keys()))
print()
print("Routing logic (check_done):")
mock_state_mid  = {"current_idx": 1, "sub_problems": ["a", "b", "c"], "solutions": ["s1"], "problem": "", "final_answer": ""}
mock_state_done = {"current_idx": 3, "sub_problems": ["a", "b", "c"], "solutions": ["s1","s2","s3"], "problem": "", "final_answer": ""}
print(f"  idx=1 of 3 sub-problems -> route: '{check_done(mock_state_mid)}'")
print(f"  idx=3 of 3 sub-problems -> route: '{check_done(mock_state_done)}'")
print()
print("Graph topology:")
print("  START -> decompose_node -> solve_next")
print("  solve_next -> [check_done] -> solve_next  (loop)")
print("                             -> finalize    -> END")


---
## Part 4 — Context Chaining: What Each `solve_next` Call Sees

This is the **key insight** of least-to-most prompting: each `solve_next` invocation sees not just the current sub-problem, but the **full problem plus all prior (sub-problem, answer) pairs**.

As we progress through the sub-problems, the context grows:
- Step 1: sees only the original problem
- Step 2: sees the problem + Step 1's answer
- Step 3: sees the problem + Steps 1 and 2's answers
- ...

This means later sub-problems can **reference concrete numbers** from earlier answers rather than re-deriving them. For example, Step 3 might say "using the $1.50 remaining from Step 2..."

### Solve Prompt Template

```
Problem: {problem}

{prior_context}Now solve this sub-problem:
{current_sub_problem}

Answer:
```


In [ ]:
SOLVE_PROMPT = '''Problem: {problem}

{prior_context}Now solve this sub-problem:
{current_sub_problem}

Answer:'''

def build_prior_context(sub_problems: list[str], solutions: list[str]) -> str:
    """Build the growing context string from prior (question, answer) pairs."""
    if not solutions:
        return ""
    lines = ["Prior solutions:\n"]
    for sp, sol in zip(sub_problems[:len(solutions)], solutions):
        lines.append(f"  Q: {sp}\n  A: {sol}\n\n")
    return "".join(lines)

# Show how context grows across 3 steps
mock_subs = [
    "How far does the first train travel by 9am?",
    "What is the combined closing speed of both trains after 9am?",
    "How long after 9am until they meet, and what is the total distance from city A?"
]
mock_sols = ["60 miles", "60 + 80 = 140 mph"]

print("=== Context seen by solve_next at each step ===\n")
for step in range(3):
    prior = build_prior_context(mock_subs, mock_sols[:step])
    prompt = SOLVE_PROMPT.format(
        problem=test_problem,
        prior_context=prior,
        current_sub_problem=mock_subs[step]
    )
    print(f"--- Step {step+1} prompt ({len(prompt)} chars) ---")
    print(prompt[:400])
    if len(prompt) > 400:
        print("  [... truncated ...]")
    print()


---
## Part 5 — Full End-to-End Run with Step-by-Step Trace

Now we assemble all nodes into the complete LangGraph pipeline:

1. **`decompose_node`**: calls `decompose()` and initializes `sub_problems`, `current_idx=0`, `solutions=[]`
2. **`solve_next`**: solves the next sub-problem using accumulated context, increments `current_idx`
3. **`check_done`**: conditional router — loops back or routes to `finalize`
4. **`finalize`**: copies the last solution into `final_answer`


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def decompose_node(state: LtMState) -> dict:
    """Decompose the problem into ordered sub-problems."""
    sub_probs = decompose(state["problem"])
    print(f"Decomposed into {len(sub_probs)} sub-problems:")
    for i, sp in enumerate(sub_probs):
        print(f"  {i+1}. {sp}")
    return {"sub_problems": sub_probs, "current_idx": 0, "solutions": []}

def solve_next(state: LtMState) -> dict:
    """Solve the next sub-problem using all prior solutions as context."""
    idx = state["current_idx"]
    sub_problem = state["sub_problems"][idx]
    prior_ctx = build_prior_context(state["sub_problems"], state["solutions"])
    prompt = SOLVE_PROMPT.format(
        problem=state["problem"],
        prior_context=prior_ctx,
        current_sub_problem=sub_problem
    )
    resp = llm.invoke([HumanMessage(content=prompt)])
    answer = resp.content.strip()
    print(f"  Step {idx+1}/{len(state['sub_problems'])}: '{sub_problem[:60]}...' -> '{answer[:60]}'")
    return {
        "solutions": state["solutions"] + [answer],
        "current_idx": idx + 1
    }

def finalize(state: LtMState) -> dict:
    """The last solution IS the final answer for this pipeline."""
    return {"final_answer": state["solutions"][-1] if state["solutions"] else ""}

# Build graph
g = StateGraph(LtMState)
g.add_node("decompose_node", decompose_node)
g.add_node("solve_next", solve_next)
g.add_node("finalize", finalize)
g.set_entry_point("decompose_node")
g.add_edge("decompose_node", "solve_next")
g.add_conditional_edges("solve_next", check_done, {"continue": "solve_next", "done": "finalize"})
g.add_edge("finalize", END)
app = g.compile()

print(f"Problem: {test_problem}\n")
result = app.invoke({"problem": test_problem, "sub_problems": [], "solutions": [], "current_idx": 0, "final_answer": ""})
print(f"\nFinal answer: {result['final_answer']}")
print(f"Solutions trace: {result['solutions']}")


---
## Part 6 — Exercises

### Exercise (a) — Inject a 4th Sub-Problem Manually

The `decompose_node` returns a list, but you can also **pre-populate or patch `sub_problems`** in the initial state before calling `app.invoke()`. This lets you test the solver on a known decomposition without burning LLM tokens on decompose.

**Task**: Add a 4th sub-problem to the train problem — for example, "What time do the trains meet?" — and verify that the solver produces 4 sequential answers.

### Exercise (b) — A Problem That Fails Without Least-to-Most

**Task**: Pick a 4-step arithmetic word problem and run it twice:
1. As a plain CoT prompt (single `llm.invoke()` call)
2. Through the full least-to-most pipeline

Compare the accuracy of the final answers. Least-to-most typically handles 4+ step problems more reliably because intermediate results are never skipped or conflated.


In [ ]:
# Exercise (a) — Inject a 4th sub-problem manually
# -------------------------------------------------------
# Instead of running decompose_node (which calls the LLM),
# pre-populate sub_problems in the initial state.
# The graph will skip decompose and jump straight to solve_next.
# (Or: pass sub_problems directly and set entry point to solve_next.)

custom_sub_problems = [
    "How far does the first train travel by 9am?",
    "What is the combined closing speed of both trains after 9am?",
    "How many hours after 9am until both trains meet?",
    # TODO: add your 4th sub-problem here
    # "What time do the trains meet?",
]

print("Exercise (a): run app.invoke() with custom sub_problems pre-populated")
print("Uncomment the 4th sub-problem and the invoke call below when ready.\n")

# result_a = app.invoke({
#     "problem": test_problem,
#     "sub_problems": custom_sub_problems,
#     "solutions": [],
#     "current_idx": 0,
#     "final_answer": ""
# })
# print(f"Final answer: {result_a['final_answer']}")
# print(f"All solutions: {result_a['solutions']}")


# Exercise (b) — Compare CoT vs least-to-most on a hard multi-step problem
# -------------------------------------------------------
hard_problem = (
    "A factory produces 240 widgets per day. It operates 5 days a week. "
    "Each widget sells for $12. The factory has fixed costs of $8,000/week "
    "and variable costs of $5 per widget. What is the weekly profit?"
)

print("\nExercise (b): hard multi-step problem")
print(f"Problem: {hard_problem}\n")

# --- Approach 1: Standard CoT (single prompt) ---
COT_PROMPT = "Solve this step by step:\n\n{problem}\n\nAnswer:"
print("TODO: run Approach 1 — single CoT call")
# cot_resp = llm.invoke([HumanMessage(content=COT_PROMPT.format(problem=hard_problem))])
# print(f"CoT answer: {cot_resp.content.strip()}")

# --- Approach 2: Least-to-most pipeline ---
print("TODO: run Approach 2 — full least-to-most pipeline")
# result_b = app.invoke({"problem": hard_problem, "sub_problems": [], "solutions": [], "current_idx": 0, "final_answer": ""})
# print(f"Least-to-most answer: {result_b['final_answer']}")


---
## Answer Key

### Exercise (a) — Injecting a 4th Sub-Problem

You can pre-populate `sub_problems` in two ways:

**Option 1 — Initial state injection** (used in the exercise starter):
Pass `sub_problems=custom_sub_problems` directly into `app.invoke()`. The `decompose_node` will overwrite this, so you need to either skip it (use a different entry point) or patch `decompose_node` to skip the LLM call when `sub_problems` is already set.

**Option 2 — Patch the decompose node**:
```python
def decompose_node(state: LtMState) -> dict:
    if state["sub_problems"]:  # pre-populated — skip LLM call
        return {"current_idx": 0, "solutions": []}
    sub_probs = decompose(state["problem"])
    return {"sub_problems": sub_probs, "current_idx": 0, "solutions": []}
```

Adding a 4th sub-problem increases the number of `solve_next` iterations by one and grows the prior context by one (Q, A) pair. The solver correctly chains all four answers.

---

### Exercise (b) — CoT vs Least-to-Most Comparison

For the factory profit problem, the correct answer is:
- Weekly widgets: 240 × 5 = **1,200 widgets**
- Weekly revenue: 1,200 × $12 = **$14,400**
- Weekly variable cost: 1,200 × $5 = **$6,000**
- Weekly total cost: $6,000 + $8,000 = **$14,000**
- Weekly profit: $14,400 − $14,000 = **$400**

**Standard CoT** often gets this right on simple problems, but on problems with 4+ interdependent steps, it may conflate intermediate values or skip sub-calculations entirely.

**Least-to-most** is more reliable because:
1. Each intermediate value is computed in isolation and stored explicitly.
2. Later steps reference the *stored concrete number* (e.g., "using the 1,200 widgets from Step 1...") rather than re-deriving it.
3. Errors are isolated — a mistake in Step 2 doesn't cascade unless the Step 3 prompt also makes an error.

The trade-off: least-to-most requires N+1 LLM calls (1 decompose + N solves) vs 1 call for CoT. For problems with 3 or fewer steps, CoT is usually sufficient and cheaper.
